## Libraries

In [52]:
from nltk.probability import FreqDist
import glob
import re
import unicodedata
import json

## Importing Docs

In [53]:
DIARIES = glob.glob("../Test_Files/Clinical_diaries/inconsistancy-diary_patient_*.txt")
TRIALS = glob.glob("../Test_Files/Clinical_trials/clinical-trial_*.txt")

EXTRACTED_CRITERION = glob.glob("../Test_Files/Clinical_trials/Criteria_extracted/clinical-trial-extracted_e*.txt")

print(f"Diaries: {len(DIARIES)}")
print(f"Trials: {len(TRIALS)}")
print(f"Extracted: {len(EXTRACTED_CRITERION)}")

Diaries: 30
Trials: 30
Extracted: 30


## Pre-Processing

In [54]:
def normalize_trials(text):
    text = normalize_text(text)

    inclusion_match = re.search(
        r"(Inclusion Criteria\s*:?\s*)(.*?)(?=Exclusion Criteria\s*:?)",
        text,
        re.IGNORECASE | re.DOTALL,
    )

    exclusion_match = re.search(
        r"(Exclusion Criteria\s*:?\s*)(.*?)(?=\n(?:Study Plan|Study Design|Investigational Product|Control Product|Study Endpoints|Primary Endpoint|Secondary Endpoints|Safety Endpoints|Follow-Up|Statistical Analysis|References)\b|\Z)",
        text,
        re.IGNORECASE | re.DOTALL,
    )

    if inclusion_match and exclusion_match:

        inclusion_text = inclusion_match.group(2).strip()
        exclusion_text = exclusion_match.group(2).strip()

        text = (
            "Inclusion Criteria:\n"
            f"{inclusion_text}\n\n"
            "Exclusion Criteria:\n"
            f"{exclusion_text}"
        )

    return text

def normalize_diaries(doc_content):
    text = normalize_text(doc_content)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    
    cleaned = []
    
    skip_patterns = [
        r"^UNIDADE LOCAL DE SAÚDE",
        r"^Diário Clínico$",
        r"^\d{2}-\d{2}-\d{4}",
        r"^(?:Dr|Dra|Dr\(a\))\.?\s+.*",
        r"^Processado por computador",
        r"^Pag\.\s*\d+/\d+",
    ]
    
    for line in lines:

        should_skip = any(
            re.search(pattern, line, re.IGNORECASE)
            for pattern in skip_patterns
        )
        if not should_skip:
            cleaned.append(line)
            
    text = "\n".join(cleaned)
    
    
    return text

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    
    text = re.sub(r"[‐-‒–—]", "-", text)

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r"\r\n?", "\n", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

## Counting Tokens

In [55]:
total_tokens_diaries = 0
num_diaries = len(DIARIES)
avg_tokens_diaries = 0

total_tokens_trials = 0
num_trials = len(TRIALS)
avg_tokens_trials = 0


for diary in DIARIES:
    with open(diary, "r", encoding="utf-8") as d:
        diary_text = d.read()

        normalized_diary = normalize_diaries(diary_text)
        
        fdist = FreqDist(normalized_diary)

        total_tokens_diaries += fdist.N()

avg_tokens_diaries = (total_tokens_diaries)/num_diaries

for trial in TRIALS:
    with open(trial, "r", encoding="utf-8") as t:
        trial_text = t.read()

        normalized_trial = normalize_trials(trial_text)

        fdist = FreqDist(normalized_trial)

        total_tokens_trials += fdist.N()

avg_tokens_trials = (total_tokens_trials)/num_trials



print("DIARY METRICS")
print(f"Total number of tokens in diaries: {total_tokens_diaries}\n")
print(f"Average of tokens in diaries: {round(avg_tokens_diaries, 0)}\n\n")

print("TRIAL METRICS")
print(f"Total number of tokens in trials: {total_tokens_trials}\n")
print(f"Average of tokens in trials: {round(avg_tokens_trials,0)}\n\n")

DIARY METRICS
Total number of tokens in diaries: 64953

Average of tokens in diaries: 2165.0


TRIAL METRICS
Total number of tokens in trials: 120786

Average of tokens in trials: 4026.0




## Counting Criteria

In [56]:
total_num_inclusion = 0
total_num_exclusion = 0
total_num = 0

avg_iclusion = 0
avg_exclusion = 0
avg_total=0

num_extracted = len(EXTRACTED_CRITERION)

for criteria in EXTRACTED_CRITERION:
    with open(criteria, "r", encoding="utf-8") as c:
        criteria_data = json.load(c)

        print(f"METRICS OF TRIAL {criteria}")
        print(f"Criteria:\n\tInclusion - {len(criteria_data['inclusion_criteria'])}\n\tExclusion - {len(criteria_data['exclusion_criteria'])}\n\tTotal - {len(criteria_data['inclusion_criteria']) + len(criteria_data['exclusion_criteria'])}")

        total_num_inclusion += len(criteria_data['inclusion_criteria'])
        total_num_exclusion += len(criteria_data['exclusion_criteria'])
        total_num += len(criteria_data['inclusion_criteria']) + len(criteria_data['exclusion_criteria'])

avg_iclusion = total_num_inclusion/num_extracted
avg_exclusion = total_num_exclusion/num_extracted
avg_total = total_num/num_extracted

print("CRITERION METRICS")
print(f"Inclusion criteria:\n\tTotal - {total_num_inclusion};\n\tAverage - {round(avg_iclusion,0)}\n")
print(f"Exclusion criteria:\n\tTotal - {total_num_exclusion};\n\tAverage - {round(avg_exclusion,0)}\n")
print(f"Aggregated:\n\tTotal - {total_num};\n\tAverage - {round(avg_total,0)}\n")

METRICS OF TRIAL ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e1.txt
Criteria:
	Inclusion - 14
	Exclusion - 11
	Total - 25
METRICS OF TRIAL ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e10.txt
Criteria:
	Inclusion - 11
	Exclusion - 16
	Total - 27
METRICS OF TRIAL ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e11.txt
Criteria:
	Inclusion - 28
	Exclusion - 5
	Total - 33
METRICS OF TRIAL ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e12.txt
Criteria:
	Inclusion - 8
	Exclusion - 11
	Total - 19
METRICS OF TRIAL ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e13.txt
Criteria:
	Inclusion - 7
	Exclusion - 9
	Total - 16
METRICS OF TRIAL ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extracted_e14.txt
Criteria:
	Inclusion - 25
	Exclusion - 13
	Total - 38
METRICS OF TRIAL ../Test_Files/Clinical_trials/Criteria_extracted\clinical-trial-extrac